# Project 1
### DecodeLabs | Data Analytics Internship
---

**Cleaning Roadmap:**
1. Load Raw Dataset
2. Identify Missing Values
3. Remove Duplicates
4. Correct Data Formats
5. Validate Logical Consistency
6. Final dtype Summary
7. Save Cleaned Dataset


## 1. Load Raw Dataset


In [1]:
import pandas as pd
import numpy as np


OUTPUT_FILE = "cleaned_dataset_v1.csv"

df = pd.read_csv(r"C:\Hazem\DecodeLabs\Project1\Dataset for Data Analytics - Sheet1.csv")

print("===============================" )
print("RAW DATASET REPORT")
print("===============================" )
print(f"  Rows      : {df.shape[0]}")
print(f"  Columns   : {df.shape[1]}")
print(f"  Columns   : {df.columns.tolist()}")


RAW DATASET REPORT
  Rows      : 1200
  Columns   : 14
  Columns   : ['OrderID', 'Date', 'CustomerID', 'Product', 'Quantity', 'UnitPrice', 'ShippingAddress', 'PaymentMethod', 'OrderStatus', 'TrackingNumber', 'ItemsInCart', 'CouponCode', 'ReferralSource', 'TotalPrice']


In [2]:
#Preview
df.head()


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [3]:
#dtype overview
df.dtypes


OrderID             object
Date                object
CustomerID          object
Product             object
Quantity             int64
UnitPrice          float64
ShippingAddress     object
PaymentMethod       object
OrderStatus         object
TrackingNumber      object
ItemsInCart          int64
CouponCode          object
ReferralSource      object
TotalPrice         float64
dtype: object

## 2. Identify Missing Values
> Detect nulls across every column and report them as counts and percentages.


In [4]:
print("======================")
print("MISSING VALUES")
print("======================" )

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
missing_report = missing_report[missing_report["Missing Count"] > 0]

if missing_report.empty:
    print("No missing values found")
else:
    print(missing_report.to_string())


MISSING VALUES
            Missing Count  Missing %
CouponCode            309      25.75


In [5]:
#Fill missing CouponCode with 'NONE'
if "CouponCode" in df.columns:
    before = df["CouponCode"].isnull().sum()
    df["CouponCode"] = df["CouponCode"].fillna("NONE")
    print(f" CouponCode: filled {before} nulls to 'NONE' ")


 CouponCode: filled 309 nulls to 'NONE' 


## 3. Remove Duplicates
> Check for fully duplicated rows and duplicate OrderIDs, then remove them.


In [6]:
print("===================" )
print("DUPLICATE ROWS")
print("===================" )

full_dupes  = df.duplicated().sum()
order_dupes = df["OrderID"].duplicated().sum()

print(f"  Full duplicate rows : {full_dupes}")
print(f"  Duplicate OrderIDs  : {order_dupes}")

before_len = len(df)
df = df.drop_duplicates()
df = df.drop_duplicates(subset=["OrderID"], keep="first")
removed = before_len - len(df)
print(f" Removed {removed} duplicate row(s) and Rows remaining: {len(df)}")


DUPLICATE ROWS
  Full duplicate rows : 0
  Duplicate OrderIDs  : 0
 Removed 0 duplicate row(s) and Rows remaining: 1200


## 4. Correct Data Formats
> Convert dates, coerce numerics, normalise text casing, and enforce ID string types.


In [7]:
print("=================================" )
print("DATA FORMAT CORRECTIONS")
print("==================================" )

#Date to datetime
df["Date"]  = pd.to_datetime(df["Date"], errors="coerce")
bad_dates   = df["Date"].isnull().sum()
print(f"  Date column converted to datetime. Unparseable dates: {bad_dates}")


DATA FORMAT CORRECTIONS
  Date column converted to datetime. Unparseable dates: 0


In [8]:
#Numeric columns – coerce strings / stray characters
for col in ["Quantity", "UnitPrice", "TotalPrice", "ItemsInCart"]:
    before_nulls = df[col].isnull().sum()
    df[col]      = pd.to_numeric(df[col], errors="coerce")
    after_nulls  = df[col].isnull().sum()
    new_nulls    = after_nulls - before_nulls
    print(f"  {col}: coerced to numeric. New NaNs introduced: {new_nulls}")


  Quantity: coerced to numeric. New NaNs introduced: 0
  UnitPrice: coerced to numeric. New NaNs introduced: 0
  TotalPrice: coerced to numeric. New NaNs introduced: 0
  ItemsInCart: coerced to numeric. New NaNs introduced: 0


In [9]:
#Text columns – strip whitespace & normalise case
text_cols = ["Product", "PaymentMethod", "OrderStatus", "ReferralSource", "CouponCode"]
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()
print(f"  Text columns stripped & title-cased: {text_cols}")

#ID columns – enforce string type
for col in ["OrderID", "CustomerID", "TrackingNumber"]:
    df[col] = df[col].astype(str).str.strip()
print(f"  ID columns enforced as strings: ['OrderID', 'CustomerID', 'TrackingNumber']")


  Text columns stripped & title-cased: ['Product', 'PaymentMethod', 'OrderStatus', 'ReferralSource', 'CouponCode']
  ID columns enforced as strings: ['OrderID', 'CustomerID', 'TrackingNumber']


## 5. Validate Logical Consistency
> Verify that `TotalPrice = Quantity × UnitPrice` and that no quantities or prices are zero/negative.


In [10]:
print("=================================" )
print("LOGICAL CONSISTENCY CHECKS")
print("=================================")

#TotalPrice should equal Quantity × UnitPrice
df["_expected_total"] = (df["Quantity"] * df["UnitPrice"]).round(2)
mismatch = df[abs(df["TotalPrice"] - df["_expected_total"]) > 0.05]
print(f"  TotalPrice ≠ Qty × UnitPrice : {len(mismatch)} row(s)")
if len(mismatch) > 0:
    print(mismatch[["OrderID","Quantity","UnitPrice","TotalPrice","_expected_total"]].head())

#Negative or zero quantities / prices
neg_qty   = (df["Quantity"]  <= 0).sum()
neg_price = (df["UnitPrice"] <= 0).sum()
print(f"  Non-positive Quantity  : {neg_qty}")
print(f"  Non-positive UnitPrice : {neg_price}")

# Drop helper column
df.drop(columns=["_expected_total"], inplace=True)


LOGICAL CONSISTENCY CHECKS
  TotalPrice ≠ Qty × UnitPrice : 0 row(s)
  Non-positive Quantity  : 0
  Non-positive UnitPrice : 0


## 6. Final Column dtype Summary
> Confirm every column now has the correct data type after all fixes.


In [11]:
print("==============================")
print("FINAL COLUMN DTYPES")
print("==============================")
print(df.dtypes.to_string())


FINAL COLUMN DTYPES
OrderID                    object
Date               datetime64[ns]
CustomerID                 object
Product                    object
Quantity                    int64
UnitPrice                 float64
ShippingAddress            object
PaymentMethod              object
OrderStatus                object
TrackingNumber             object
ItemsInCart                 int64
CouponCode                 object
ReferralSource             object
TotalPrice                float64


In [12]:
# Final preview of cleaned data
df.head()


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,Save10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,Save10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,Freeship,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,Save10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,Save10,Email,2504.04


## 7. Save Cleaned Dataset
> Export the cleaned DataFrame to a CSV file ready for Project 2 EDA.


In [13]:
df.to_csv(OUTPUT_FILE, index=False)

print("=========================================")
print(f"CLEANED DATASET SAVED → {OUTPUT_FILE}")
print(f"  Final shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("===========================================")


CLEANED DATASET SAVED → cleaned_dataset_v1.csv
  Final shape: 1200 rows × 14 columns
